# Fraud Detection

This notebook builds a real-time fraud scoring pipeline with `decider`.

Two modules are composed with `|`:

- **VelocityFeatures** — normalises transaction volume and amount against baselines
- **FraudScorer** — combines the two signals into a `fraud_score` and a binary `is_fraud` flag

The blending weights are controlled by `FraudConfig` and can be updated without touching the function logic.

In [ ]:
%pip install -q decider


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import polars as pl

In [ ]:
from decider.modules.functional import generate_from_functions
from decider.modules._ext import register_graph_module, GraphModule
from pydantic import BaseModel

## Sample transactions

Four transactions: `txn_id`, raw `amount`, rolling `avg_amount`, and hourly `txn_count`.

In [ ]:
transactions = pl.DataFrame({
    "txn_id":     ["T001", "T002", "T003", "T004"],
    "amount":     [120.0,  5_500.0, 45.0,  980.0],
    "avg_amount": [100.0,    200.0, 50.0,  300.0],
    "txn_count":  [3,         48,    1,      12],
})

print(transactions)

## Module 1 — VelocityFeatures

`txn_velocity` normalises the transaction count to an hourly rate.  
`amount_spike` measures how far the current amount deviates from the customer average.

In [ ]:
def txn_velocity(txn_count: pl.Expr) -> pl.Expr:
    return txn_count / 24

def amount_spike(amount: pl.Expr, avg_amount: pl.Expr) -> pl.Expr:
    return amount / avg_amount

VelocityFeatures = generate_from_functions("velocity_features", txn_velocity, amount_spike)
register_graph_module(VelocityFeatures)
velocity_features = VelocityFeatures(name="velocity_features")

print(velocity_features({"input": transactions}))

## Module 2 — FraudScorer

`fraud_score` is a weighted combination of the two signals.  
`is_fraud` fires when the score exceeds `1.0` — meaning the transaction looks anomalous on at least one dimension.

In [ ]:
class FraudConfig(BaseModel):
    velocity_weight: float = 0.6
    spike_weight: float = 0.4

def fraud_score(txn_velocity: pl.Expr, amount_spike: pl.Expr, config: FraudConfig) -> pl.Expr:
    return (config.velocity_weight * txn_velocity + config.spike_weight * amount_spike).round(4)

def is_fraud(fraud_score: pl.Expr) -> pl.Expr:
    return fraud_score > 1.0

FraudScorer = generate_from_functions("fraud_scorer", fraud_score, is_fraud)
register_graph_module(FraudScorer)
fraud_scorer = FraudScorer(name="fraud_scorer", velocity_weight=0.6, spike_weight=0.4)

print(fraud_scorer)

## Composing the pipeline

Chain `VelocityFeatures | FraudScorer` so the scorer automatically receives the derived columns.

In [ ]:
pipeline = velocity_features | fraud_scorer

result = pipeline({"input": transactions})
print(result.select(["txn_id", "txn_velocity", "amount_spike", "fraud_score", "is_fraud"]))

## Serialisation to JSON

The entire pipeline — including the nested `FraudConfig` weights — serialises cleanly to JSON.
This string can be stored in a config store or version-controlled alongside model artefacts.

In [ ]:
import json

pipeline_json = json.dumps(pipeline.model_dump(), indent=2)
print(pipeline_json)

## Restore from JSON and verify

`GraphModule.model_validate()` deserialises the JSON back into a live pipeline.

In [ ]:
restored = GraphModule.model_validate(json.loads(pipeline_json)).root
restored_result = restored({"input": transactions})

print("Outputs match after roundtrip:", result.equals(restored_result))
print(restored_result.select(["txn_id", "fraud_score", "is_fraud"]))